In [ ]:
# pip install selenium  webdriver-manager

# Capstone Project

#### Project objective is to collect, clean, analyze, model, and visualize real-world data that is cricket players data from website to derive meaningful insights and communicate them through an interactive dashboard.

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

base_url = "https://stats.espncricinfo.com/ci/engine/stats/index.html"

params = {
    "class": 11,
    "template": "results",
    "type": "allround"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

# Retry session 
session = requests.Session()
retry = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

all_rows = []
headers_row = []

for page in tqdm(range(1, 135), desc="Scraping ESPN Cricinfo"):

    params["page"] = page

    try:
        response = session.get(base_url, params=params, headers=headers, timeout=30)

        soup = BeautifulSoup(response.text, "html.parser")

        tables = soup.find_all("table", class_="engineTable")

        if len(tables) < 3:
            print(f"Stats table missing on page {page}")
            continue

        table = tables[2]

        # extract headers once
        if not headers_row:
            headers_row = [th.text.strip() for th in table.find_all("th")]

        rows = table.find_all("tr", class_="data1")

        for row in rows:
            cols = [td.text.strip() for td in row.find_all("td")]
            if cols:
                cols.append(page)
                all_rows.append(cols)

        # polite delay
        time.sleep(2)

    except Exception as e:
        print(f"Error on page {page}: {e}")
        time.sleep(5)
        continue


columns = headers_row + ["Page"]

df = pd.DataFrame(all_rows, columns=columns)

df.to_csv("ESPN_AllRounders_ODI.csv", index=False)

print("Scraping Completed")
print("Total rows:", len(df))
print(df.head())

Scraping ESPN Cricinfo:  19%|██████████▋                                              | 25/134 [07:11<31:10, 17.16s/it]

Error on page 26: HTTPSConnectionPool(host='stats.espncricinfo.com', port=443): Max retries exceeded with url: /ci/engine/stats/index.html?class=11&template=results&type=allround&page=26 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002134ACBC910>: Failed to resolve 'stats.espncricinfo.com' ([Errno 11001] getaddrinfo failed)"))


Scraping ESPN Cricinfo:  19%|██████████▋                                            | 26/134 [08:49<1:14:23, 41.33s/it]

Error on page 27: HTTPSConnectionPool(host='stats.espncricinfo.com', port=443): Max retries exceeded with url: /ci/engine/stats/index.html?class=11&template=results&type=allround&page=27 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002134ACBC910>: Failed to resolve 'stats.espncricinfo.com' ([Errno 11001] getaddrinfo failed)"))


Scraping ESPN Cricinfo: 100%|████████████████████████████████████████████████████████| 134/134 [42:49<00:00, 19.18s/it]

Scraping Completed
Total rows: 6600
                        Player       Span  Mat   Runs    HS Bat Av  100 Wkts  \
0           SR Tendulkar (IND)  1989-2013  664  34357  248*  48.52  100  201   
1   DPMD Jayawardene (Asia/SL)  1997-2015  652  25957   374  39.15   54   14   
2  KC Sangakkara (Asia/ICC/SL)  2000-2015  594  28016   319  46.77   63    0   
3      ST Jayasuriya (Asia/SL)  1989-2011  586  21032   340  34.14   42  440   
4         RT Ponting (AUS/ICC)  1995-2012  560  27483   257  45.95   71    8   

    BBI Bowl Av  5   Ct   St Ave Diff    Page  
0  5/32   46.53  2  256    0     1.98       1  
1  2/32   62.92  0  440    0   -23.77       1  
2     -       -  0  609  139        -       1  
3  6/29   35.66  6  205    0    -1.52       1  
4   1/0    47.5  0  364    0    -1.54       1  


In [6]:
df.shape

(6600, 16)

In [7]:
df.dtypes

Player      object
Span        object
Mat         object
Runs        object
HS          object
Bat Av      object
100         object
Wkts        object
BBI         object
Bowl Av     object
5           object
Ct          object
St          object
Ave Diff    object
            object
Page         int64
dtype: object

### Problem Statement

In modern cricket analytics, evaluating the true contribution of all-rounders is challenging because their performance depends on both batting and bowling abilities. Traditional statistics such as total runs or wickets alone do not accurately represent a player’s overall impact on a match or tournament.

The objective of this project is to develop a data-driven analytics model to evaluate and analyse cricket all-rounders based on their combined batting and bowling performance. Using publicly available cricket statistics, the project aims to collect, clean, and analyze player data to compute an All-Rounder Performance Score that reflects overall contribution.

### Data Cleaning & Preprocessing

In [10]:
df.drop("Page",axis=1,inplace=True)

In [16]:
#Renaming the columns
df.rename(columns={"span":"Career_span","Mat":"Matches","HS":"Highest_score","Bat Av":"Batting_Avg","100":"Centuries","Wkts":"Wickets",
                   "Bowl Avg":"Bowling_Avg",
    "5":"Five_Wicket_Hauls","Ct":"Catches","St":"Stumpings","Avg Diff":"Avg_Diff"},inplace=True)

In [17]:
df.columns

Index(['Player', 'Span', 'Matches', 'Runs', 'Highest_score', 'Batting_Avg',
       'Centuries', 'Wickets', 'Bowl Av', 'Five_Wicket_Hauls', 'Catches',
       'Stumpings', 'Ave Diff', ''],
      dtype='object')

In [18]:
num_cols = [
    "Matches","Runs","Highest_score","Batting_Avg","Centuries","Wickets","Bowl Av",
    "Five_Wicket_Hauls","Catches","Stumpings","Ave Diff"]

# Clean and convert
for col in num_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("*", "", regex=False)
            .str.replace("-", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [19]:
df["Player"] = df["Player"].str.replace(r"\s*\(.*?\)", "", regex=True)

In [20]:
df.head()

,Player,Span,Matches,Runs,Highest_score,Batting_Avg,Centuries,Wickets,Bowl Av,Five_Wicket_Hauls,Catches,Stumpings,Ave Diff,
0,SR Tendulkar,1989-2013,664,34357.0,248.0,48.52,100.0,201.0,46.53,2.0,256,0,1.98,
1,DPMD Jayawardene,1997-2015,652,25957.0,374.0,39.15,54.0,14.0,62.92,0.0,440,0,23.77,
2,KC Sangakkara,2000-2015,594,28016.0,319.0,46.77,63.0,0.0,NaN,0.0,609,139,NaN,
3,ST Jayasuriya,1989-2011,586,21032.0,340.0,34.14,42.0,440.0,35.66,6.0,205,0,1.52,
4,RT Ponting,1995-2012,560,27483.0,257.0,45.95,71.0,8.0,47.50,0.0,364,0,1.54,


In [21]:
df.isnull().sum()

Player                  0
Span                    0
Matches                 0
Runs                   71
Highest_score          71
Batting_Avg           270
Centuries              71
Wickets              1670
Bowl Av              2294
Five_Wicket_Hauls    1670
Catches                 0
Stumpings               0
Ave Diff             2507
                        0
dtype: int64

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6600 entries, 0 to 6599
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Player             6600 non-null   object 
 1   Span               6600 non-null   object 
 2   Matches            6600 non-null   int64  
 3   Runs               6529 non-null   float64
 4   Highest_score      6529 non-null   float64
 5   Batting_Avg        6330 non-null   float64
 6   Centuries          6529 non-null   float64
 7   Wickets            4930 non-null   float64
 8   Bowl Av            4306 non-null   float64
 9   Five_Wicket_Hauls  4930 non-null   float64
 10  Catches            6600 non-null   int64  
 11  Stumpings          6600 non-null   int64  
 12  Ave Diff           4093 non-null   float64
 13                     6600 non-null   object 
dtypes: float64(8), int64(3), object(3)
memory usage: 722.0+ KB


In [24]:
import numpy as np
df = df.replace("-", np.nan)

In [25]:
df.isnull().sum()

Player                  0
Span                    0
Matches                 0
Runs                   71
Highest_score          71
Batting_Avg           270
Centuries              71
Wickets              1670
Bowl Av              2294
Five_Wicket_Hauls    1670
Catches                 0
Stumpings               0
Ave Diff             2507
                        0
dtype: int64

In [26]:
df.fillna(0, inplace=True)
df = df.infer_objects(copy=False)

In [27]:
df = df.drop_duplicates()

In [28]:
df.isnull().sum()

Player               0
Span                 0
Matches              0
Runs                 0
Highest_score        0
Batting_Avg          0
Centuries            0
Wickets              0
Bowl Av              0
Five_Wicket_Hauls    0
Catches              0
Stumpings            0
Ave Diff             0
                     0
dtype: int64

In [31]:
df["performance_score"] = df["Batting_Avg"] - df["Bowl Av"]

In [32]:
df

,Player,Span,Matches,Runs,Highest_score,Batting_Avg,Centuries,Wickets,Bowl Av,Five_Wicket_Hauls,Catches,Stumpings,Ave Diff,,performance_score
0,SR Tendulkar,1989-2013,664,34357.0,248.0,48.52,100.0,201.0,46.53,2.0,256,0,1.98,,1.99
1,DPMD Jayawardene,1997-2015,652,25957.0,374.0,39.15,54.0,14.0,62.92,0.0,440,0,23.77,,-23.77
2,KC Sangakkara,2000-2015,594,28016.0,319.0,46.77,63.0,0.0,0.00,0.0,609,139,0.00,,46.77
3,ST Jayasuriya,1989-2011,586,21032.0,340.0,34.14,42.0,440.0,35.66,6.0,205,0,1.52,,-1.52
4,RT Ponting,1995-2012,560,27483.0,257.0,45.95,71.0,8.0,47.50,0.0,364,0,1.54,,-1.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6595,GM Guard,1958-1960,2,11.0,7.0,5.50,0.0,3.0,60.66,0.0,2,0,55.16,,-55.16
6596,RD Gudhka,2011-2011,2,17.0,10.0,17.00,0.0,1.0,60.00,0.0,0,0,42.99,,-43.00
6597,C Gunasekara,2022-2024,2,16.0,16.0,0.00,0.0,0.0,0.00,0.0,0,0,0.00,,0.00
6598,Gursharan Singh,1990-1990,2,22.0,18.0,11.00,0.0,0.0,0.00,0.0,3,0,0.00,,11.00


In [33]:
df.to_csv("clean_cricket_dataset.csv", index=False)